# Exploring Italian-culture features in Gemma

This notebook is a guided experiment in **activation steering**: identify sparse autoencoder (SAE) features that respond more strongly to Italian-culture examples than to matched controls, then add one feature's decoder direction to the model's internal activations during generation. You will compare unmodified and steered responses in **Gemma 2 2B IT**, then repeat the experiment with **Gemma 3 1B IT** and its own SAE.

The goal is to investigate a candidate feature, not to assume that one feature represents an entire culture. The supplied examples mostly concern food, routines, and places; their rankings reflect that particular dataset. A larger activation difference is a useful starting point, not proof of semantic meaning or causal importance.

**How to use this walkthrough:** run cells from top to bottom in a fresh Google Colab GPU runtime. First install dependencies and load Gemma 2, build your prompt sets, rank features, and compare baseline and steered generations. Then explore steering strengths and repeat with Gemma 3. The second half reuses your discovery and evaluation prompts, but computes a new feature ranking.

**Before starting:** enable a GPU runtime, obtain access to the Gemma model repositories on Hugging Face, and add a Colab secret named `HF_TOKEN` with repository read access. Model and SAE downloads require network access and sufficient RAM/VRAM. This notebook retains the original environment setup, including a package-file workaround; dependency versions are unpinned, so fresh environments may need compatibility adjustments. Outputs have been cleared so that your run produces its own results.

## 1. Set up the environment

Install TransformerLens (for model hooks) and SAE Lens (for loading and encoding SAE features). Run the imports next. If Colab asks you to restart after installation, restart and rerun the imports before continuing.

In [ ]:
!pip install transformer_lens sae_lens

In [ ]:
import os
import gc

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_lens import HookedTransformer
from sae_lens import SAE

### Existing memory workaround

The next cell retains the original workaround that removes a float32 conversion in TransformerLens's installed source. Its path assumes Python 3.13 in Colab; check that path against your environment before running it. It edits the installed package, and already imported code may require a runtime restart and rerunning the imports for the change to take effect. If your installed version no longer contains the target line, the message alone does not establish compatibility.

In [ ]:
file_path = "/usr/local/lib/python3.13/dist-packages/transformer_lens/weight_processing.py"

with open(file_path, "r") as f:
    code = f.read()

target = "state_dict[k] = v.float()"
replacement = "state_dict[k] = v"

if target in code:
    code = code.replace(target, replacement)
    with open(file_path, "w") as f:
        f.write(code)
    print("✓ Patch applicata con successo! Ora i pesi non vengono convertiti in float32.")
else:
    print("✓ Patch già presente o stringa già modificata.")

## 2. Load Gemma 2 and its SAE

Authenticate with Hugging Face, load the instruction-tuned Gemma 2 model on CPU, and wrap it with TransformerLens on the GPU. The configuration preserves the activation coordinates used for the experiment. Free the temporary model reference, then load the 65k-feature residual-stream SAE for layer 20 and check memory usage.

This SAE release is labeled `pt` (pretrained), while the model here is instruction-tuned. Treat transfer to this model as an experimental assumption and inspect the resulting behavior rather than assuming perfect alignment. SAE Lens versions can also differ in whether `SAE.from_pretrained` returns an SAE or a tuple; this notebook retains the original loading calls.

In [ ]:
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
model_id = "google/gemma-2-2b-it"

In [ ]:
hf_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    token=hf_token,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

In [ ]:
model = HookedTransformer.from_pretrained(
    "gemma-2-2b-it",
    hf_model=hf_model,
    tokenizer=tokenizer,
    device="cuda",
    dtype=torch.bfloat16,
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    fold_value_biases=False,
    refactor_factored_attn_matrices=False,
)

In [ ]:
del hf_model
gc.collect()
torch.cuda.empty_cache()

print(f"Allocated VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gemma-scope-2b-pt-res",            # release Gemma Scope
    sae_id="layer_20/width_65k/average_l0_61",  # hook point desiderato
    device="cuda",
)
sae = sae.to(dtype=torch.bfloat16)

In [ ]:
vram_gb = torch.cuda.memory_allocated() / 1e9
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nModello: {model.cfg.model_name}")
print(f"VRAM allocata: {vram_gb:.2f} GB / {total_vram_gb:.2f} GB (Spazio libero: {total_vram_gb - vram_gb:.2f} GB)")

### Optional memory cleanup

This cell releases cached discovery tensors and a previous exception traceback before feature extraction. It is useful after an interrupted run. If you rerun it after discovery, rerun discovery before selecting a feature.

In [ ]:
import sys
import gc
import torch

# Rilascia il traceback dell'eccezione che tiene occupata la memoria
sys.last_traceback = None
if 'target_acts' in globals():
    del target_acts
if 'control_acts' in globals():
    del control_acts

gc.collect()
torch.cuda.empty_cache()

vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"VRAM libera recuperata: {vram_free:.2f} GB")

## 3. Design your own discovery prompts

**Before reading or running the examples below, try writing 25 target prompts and 25 control prompts yourself.** Replace `italian_prompts` and `control_prompts` with your own lists, keeping those variable names so the rest of the notebook still runs. The provided 25 + 25 examples are a starting point if you want to test the pipeline first.

Choose a hypothesis: for example, do you want to study Italian food, places, everyday routines, or Italian-language text? The supplied prompts are mainly English sentences about Italian culture, so they do not isolate the Italian language. To investigate language instead, design a separate comparison and state what you changed.

1. Write 25 varied target examples spanning the aspects you want to study. Avoid repeating the same keywords throughout the list.
2. Write 25 controls with similar topics, sentence structures, and approximate lengths, while changing the property under investigation. For instance, compare an Italian meal with another meal rather than with an unrelated technical statement.
3. Keep the language consistent across sets when studying culture. Do not make one set Italian and the other English unless language is the intended variable.
4. Save a separate set of evaluation questions that you will not use to rank features. Decide beforehand what would count as a change: Italian words, cultural references, language switching, or loss of coherence.

The code subtracts the two sets' average activations; matching examples helps reduce confounds such as food vocabulary or prompt length. Each prompt's activations are **summed over token positions** before averaging across prompts, so longer prompts can contribute more. The lists are compared as groups, not as individually subtracted pairs. After editing either list, rerun discovery and every downstream steering cell in both model sections.

In [ ]:
HOOK_POINT = "blocks.20.hook_resid_post"

italian_prompts = [
    # Cibo e dogmi culinari
    "I am cooking fresh pasta with homemade tomato sauce and basil",
    "My nonna makes the best traditional lasagna every Sunday",
    "You should never put pineapple on an authentic pizza",
    "Real carbonara is made only with guanciale, pecorino, and egg yolks",
    "Breaking dry spaghetti before boiling it in salted water is forbidden",
    "Fresh extra virgin olive oil and garlic make every meal better",
    "Cream and bacon do not belong in an authentic Roman carbonara",
    "Freshly grated Parmigiano Reggiano is the final touch on warm risotto",
    "You must always cook pasta strictly al dente so it keeps its firm bite",
    "A glass of Chianti wine pairs wonderfully with seasoned cured prosciutto",
    "Dipping crispy cantucci into sweet Vin Santo at the end of the meal",

    # Rituali del caffè e aperitivo
    "I always drink an espresso at the local bar every morning",
    "Never order a cappuccino after eleven in the morning in Italy",
    "Drinking an espresso while standing at the marble counter is the proper way",
    "We met at the piazza to drink an Aperol spritz for aperitivo",
    "Nothing compares to handmade artisanal gelato on a hot summer evening",

    # Cultura, famiglia e stereotipi comportamentali
    "I talk with my hands whenever I have an animated conversation",
    "Gesticulating wildly is simply our natural way of emphasizing every sentence",
    "A proper Italian Sunday lunch easily lasts four hours with the whole family",
    "Taking an evening passeggiata along the street before dinner is a daily ritual",

    # Simboli e luoghi
    "Riding a vintage Vespa through narrow cobblestone alleys is pure freedom",
    "We walked through Florence admiring the Renaissance marble statues",
    "The sunset over the rolling Tuscan hills covered in vineyards was breathtaking",
    "Every Italian town has an ancient church standing in the central square",
    "Visiting the ancient ruins of the Colosseum in Rome was an unforgettable trip",
]

control_prompts = [
    # Cibo e cucina generica / non italiana (essenziale per non isolare la feature "food")
    "I am preparing a bowl of steamed rice with grilled chicken and broccoli",
    "My grandmother bakes a fresh apple pie with cinnamon every weekend",
    "You should never overcook a thick beef burger patty on the barbecue grill",
    "A classic sandwich is made with sliced cheddar cheese, ham, and mustard",
    "Boiling eggs for exactly seven minutes gives them a soft runny center",
    "Fresh butter and sea salt make warm sourdough bread taste much better",
    "Ketchup and processed cheese slices are common toppings on American burgers",
    "Freshly cracked black pepper is the final garnish on grilled steak",
    "You must always bake bread at high heat so the crust turns crispy and dark",
    "A cold lager beer pairs nicely with seasoned spicy barbecue wings",
    "Dipping chocolate chip cookies into cold milk before going to bed",

    # Bevande e routine quotidiane
    "I always drink a hot black tea with lemon every single morning",
    "Never leave dairy milk out on the counter at room temperature overnight",
    "Drinking a smoothie while sitting at the office desk is my normal routine",
    "We met at the park to drink cold iced lemonade after work",
    "Nothing compares to fresh vanilla frozen yogurt with chopped strawberries",

    # Comunicazione e abitudini
    "I usually take notes in my notebook whenever attending a lecture",
    "Taking deep breaths is a proven way to reduce anxiety during a presentation",
    "A standard weekend brunch with close friends usually lasts about two hours",
    "Taking a casual jog around the neighborhood before breakfast is a healthy habit",

    # Trasporti e viaggi neutri
    "Driving an electric bicycle through the city park paths is very relaxing",
    "We walked through the museum admiring modern abstract paintings on display",
    "The sunrise over the snowy mountain peaks was completely breathtaking",
    "Every quiet suburb has a modern grocery store located near the main road",
    "Visiting the modern architecture towers in downtown Chicago was an interesting trip",
]

## 4. Rank Gemma 2 feature candidates

For each prompt, capture the residual stream at `blocks.20.hook_resid_post`, encode it with the SAE, and sum each feature's activation over tokens. Compute the target and control averages, then rank features by `target average - control average`.

The table shows the ten largest differences. Inspect both the target and control columns: a large difference suggests association with your dataset, but it can still reflect a narrower topic or wording pattern. If all differences are small or negative, revisit the prompt design rather than assuming the first candidate is a strong Italian-culture feature.

In [ ]:
@torch.inference_mode()
def get_feature_activations(prompts):
    total = torch.zeros(sae.cfg.d_sae, device="cuda", dtype=torch.float32)

    for p in prompts:
        tokens = model.to_tokens(p)
        _, cache = model.run_with_cache(
            tokens,
            names_filter=[HOOK_POINT],
            stop_at_layer=21
        )
        acts = sae.encode(cache[HOOK_POINT])
        total += acts.sum(dim=1).squeeze().detach()
        del cache, acts

    return total / len(prompts)


target_acts = get_feature_activations(italian_prompts)
control_acts = get_feature_activations(control_prompts)
diff = target_acts - control_acts

top_vals, top_idxs = diff.topk(10)
print("Top Italian-specific features (65k SAE, layer 20):")
print(f"{'Feature':>10} {'Target':>10} {'Control':>10} {'Diff':>10}")
print("-" * 44)
for idx, d in zip(top_idxs, top_vals):
    print(f"{idx.item():>10} {target_acts[idx].item():>10.1f} {control_acts[idx].item():>10.1f} {d.item():>10.1f}")

## 5. Choose a feature and evaluation questions

The next cell **selects the first feature from the ranking you just computed**, replacing the fixed feature ID from the original notebook. `FEATURE_RANK = 0` means the top candidate; use another rank from 0 to 9 to investigate an alternative. Feature IDs belong to this specific SAE and must be rediscovered when the model or SAE changes.

The hook adds `STRENGTH * feature_dir` to the residual stream at every token position. `STRENGTH = 350` is an initial setting to explore, not a calibrated optimum. `TEMP` controls generation sampling.

**Create seven evaluation prompts of your own before running the supplied seven examples.** Use open questions that do not explicitly ask for Italian output or mention the discovery examples. Include varied tasks, such as explanation, creative writing, advice, and self-description. Keep this list fixed between baseline and steering and reuse it for Gemma 3. This makes it easier to see whether the intervention generalizes beyond the sentences used to select the feature.

In [ ]:
# Select the highest-scoring feature from this Gemma 2 discovery run.
# Change FEATURE_RANK (0 = first, 1 = second, ...) to test another candidate.
FEATURE_RANK = 0
ITALIAN_FEATURE = int(top_idxs[FEATURE_RANK].item())
STRENGTH = 350
TEMP = 0.3

from functools import partial
def steering_hook(resid, hook, feature_dir, strength):
    resid[:, :, :] += strength * feature_dir
    return resid

prompts = [
    "Tell me about yourself.",
    "What is the most important technology?",
    "What are your hobbies?",
    "Write a poem about the ocean.",
    "What should I eat for dinner tonight?",
    "What is your name?",
    "What is the meaning of life?",
]

feature_dir = sae.W_dec[ITALIAN_FEATURE]
hook_fn = partial(steering_hook, feature_dir=feature_dir, strength=STRENGTH)

print(f"Selected Gemma 2 candidate: {ITALIAN_FEATURE} (rank {FEATURE_RANK + 1})")


## 6. Compare baseline and steered Gemma 2 responses

Run the baseline cell without a steering hook, then run the steered cell on the same seven questions. Compare cultural references, language, relevance, repetition, and factual quality. These Gemma 2 calls use raw prompt completion and do not reset a random seed, so sampling variation can also cause differences. Repeated runs and the zero-strength control in the next section help you judge how stable a pattern is.

In [ ]:
from IPython.display import display, Markdown

def strip_prompt(output, prompt):
    """Strip the prompt prefix from model.generate() output."""
    if isinstance(output, str) and prompt in output:
        output = output[output.index(prompt) + len(prompt):]
    return output.lstrip("\n")

display(Markdown("## BASELINE (no steering)\n---"))
for prompt in prompts:
    output = model.generate(prompt, max_new_tokens=120, temperature=TEMP, verbose=False)
    output = strip_prompt(output, prompt)
    display(Markdown(f"---\n**> {prompt}**\n\n{output.strip()}"))

In [ ]:
display(Markdown(f"## STEERED (feature {ITALIAN_FEATURE}, strength={STRENGTH})\n---"))
for prompt in prompts:
    with model.hooks(fwd_hooks=[(HOOK_POINT, hook_fn)]):
        output = model.generate(prompt, max_new_tokens=120, temperature=TEMP, verbose=False)
    output = strip_prompt(output, prompt)
    display(Markdown(f"---\n**> {prompt}**\n\n{output.strip()}"))

## 7. Sweep Gemma 2 steering strengths

Hold the feature and question fixed while varying the intervention strength. Zero is the no-intervention control; larger values test how behavior changes as the direction is amplified. Look for a useful range before repetition or incoherence appears. Try more than one of your evaluation questions before drawing a conclusion. This sweep uses a separate hook variable so rerunning the earlier steered comparison keeps its configured strength.

In [ ]:
prompt = "Tell me about yourself."

for strength in [0, 50, 150, 250, 350, 500, 700, 1000]:
    sweep_hook = partial(steering_hook, feature_dir=sae.W_dec[ITALIAN_FEATURE], strength=strength)
    with model.hooks(fwd_hooks=[(HOOK_POINT, sweep_hook)]):
        output = model.generate(prompt, max_new_tokens=100, temperature=TEMP, verbose=False)
    output = strip_prompt(output, prompt)
    display(Markdown(f"### strength={strength}\n\n{output.strip()}"))

## 8. Switch to Gemma 3

The next cells release the Gemma 2 model and load Gemma 3 1B IT with a different SAE at layer 22. Finish your Gemma 2 comparisons before proceeding: once these variables are replaced, rerun the Gemma 2 loading and discovery cells if you want to return to it.

The discovery lists and the seven evaluation prompts remain available for reuse. Feature IDs and steering strengths are specific to each SAE, so do not interpret an equal numeric strength as an equal intervention across models.

In [ ]:
import gc
import torch

# Delete previous model instances and tensors from namespace
for var_name in ["model", "hf_model", "sae", "target_acts", "control_acts"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
torch.cuda.empty_cache()

free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"VRAM cleared successfully. Free space: {free_gb:.2f} GB")

In [ ]:
model_id = "google/gemma-3-1b-it"

hf_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="cpu",  # Load on CPU first to prevent VRAM spikes on Colab T4
    token=hf_token,
    low_cpu_mem_usage=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)

model = HookedTransformer.from_pretrained(
    "gemma-3-1b-it",
    hf_model=hf_model,
    tokenizer=tokenizer,
    device="cuda",
    dtype=torch.bfloat16,
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
    fold_value_biases=False,
    refactor_factored_attn_matrices=False,
)

# Free host RAM immediately
del hf_model
gc.collect()
torch.cuda.empty_cache()

print(f"Gemma 3 1B loaded! VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
HOOK_POINT = "blocks.22.hook_resid_post"

sae = SAE.from_pretrained(
    release="gemma-scope-2-1b-it-res",
    sae_id="layer_22_width_65k_l0_medium",
    device="cuda",
).to(dtype=next(model.parameters()).dtype)

vram_gb = torch.cuda.memory_allocated() / 1e9
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"SAE loaded at {HOOK_POINT}!")
print(
    f"VRAM allocated: {vram_gb:.2f} GB / {total_vram_gb:.2f} GB "
    f"(Free: {total_vram_gb - vram_gb:.2f} GB)"
)

### Capture the correct layer

Layer indices start at zero: to capture block 22, the forward pass must include that block and stop at layer 23. The extraction function below derives this stopping point directly from `HOOK_POINT`; the standalone value is a reminder.

In [ ]:
stop_at_layer=23  # Includes block 22; layer indices start at zero.

## 9. Rediscover feature candidates in Gemma 3

Run the same target-versus-control comparison on your existing 25 + 25 discovery prompts, using Gemma 3's SAE. This implementation accumulates token sums on CPU in float32 and derives the stopping layer from the hook name. The statistic matches the Gemma 2 experiment, including its sensitivity to prompt length.

Inspect the new table independently: the first candidate need not have the same meaning as Gemma 2's first candidate, even when the prompt sets are identical.

In [ ]:
@torch.inference_mode()
def get_feature_activations(prompts):
    if not prompts:
        raise ValueError("The prompt list must not be empty.")

    total = torch.zeros(sae.cfg.d_sae, dtype=torch.float32, device="cpu")
    layer = int(HOOK_POINT.split(".")[1])

    for p in prompts:
        # Keep raw-text discovery, matching your Gemma 2 experiment.
        tokens = model.to_tokens(p)
        _, cache = model.run_with_cache(
            tokens,
            names_filter=[HOOK_POINT],
            stop_at_layer=layer + 1,  # 23 for block 22
            return_type=None,
        )

        resid = cache[HOOK_POINT].to(
            device=sae.W_dec.device,
            dtype=sae.W_dec.dtype,
        )
        acts = sae.encode(resid)

        # Same token-sum statistic as before, accumulated in float32.
        total += acts.float().sum(dim=1)[0].cpu()
        del tokens, cache, resid, acts

    return total / len(prompts)


target_acts = get_feature_activations(italian_prompts)
control_acts = get_feature_activations(control_prompts)
diff = target_acts - control_acts

top_vals, top_idxs = diff.topk(10)
print(f"Top Italian-culture feature candidates ({HOOK_POINT}):")
print(f"{'Feature':>10} {'Target':>10} {'Control':>10} {'Diff':>10}")
print("-" * 44)

for idx, d in zip(top_idxs.tolist(), top_vals.tolist()):
    print(
        f"{idx:>10} {target_acts[idx].item():>10.1f} "
        f"{control_acts[idx].item():>10.1f} {d:>10.1f}"
    )

## 10. Configure Gemma 3 steering

Select a candidate from the newly computed ranking with `FEATURE_RANK`, then build its decoder direction and steering hook. The seven evaluation prompts from the Gemma 2 section are reused automatically, including any edits you made. The hook casts the direction to the residual stream's device and dtype before adding it.

In [ ]:
# Select from Gemma 3's own ranking; feature IDs do not transfer across SAEs.
FEATURE_RANK = 0
ITALIAN_FEATURE = int(top_idxs[FEATURE_RANK].item())
STRENGTH = 350
TEMP = 0.3

from functools import partial

def steering_hook(resid, hook, feature_dir, strength):
    direction = feature_dir.to(device=resid.device, dtype=resid.dtype)
    return resid + strength * direction


# Reuse the seven evaluation prompts defined in the Gemma 2 section.
# This keeps the comparison consistent when you customize your experiment.

feature_dir = sae.W_dec[ITALIAN_FEATURE].detach()
hook_fn = partial(
    steering_hook,
    feature_dir=feature_dir,
    strength=STRENGTH,
)

print(f"Selected Gemma 3 candidate: {ITALIAN_FEATURE}")

## 11. Compare Gemma 3 baseline and steered responses

The generation helper uses the tokenizer's chat template by default, stops on the configured end tokens, and returns only newly generated text. It resets the seed before each call so each baseline/steered pair starts from the same random seed; divergence can still occur once the intervention changes the token probabilities.

Run the baseline and then the steered cell. Compare each pair using the criteria you chose earlier. For a closer formatting comparison with Gemma 2, set `USE_CHAT = False` and rerun both Gemma 3 cells. Even then, model size, training, SAE, and generation details differ, so this is an exploratory comparison rather than a controlled benchmark.

In [ ]:
from IPython.display import display, Markdown

model.eval()
tokenizer = model.tokenizer
DEVICE = next(model.parameters()).device

USE_CHAT = True  # Set False to reproduce the earlier raw-completion format.
SEED = 42

stop_ids = [tokenizer.eos_token_id]
if "<end_of_turn>" in tokenizer.get_vocab():
    stop_ids.append(tokenizer.convert_tokens_to_ids("<end_of_turn>"))
stop_ids = sorted(set(x for x in stop_ids if x is not None))


@torch.inference_mode()
def generate_response(prompt, max_new_tokens=120):
    # Reset for each call so baseline/steered runs start with the same seed.
    torch.manual_seed(SEED)

    if USE_CHAT:
      encoded = tokenizer.apply_chat_template(
          [{"role": "user", "content": prompt}],
          tokenize=True,
          add_generation_prompt=True,
          return_tensors="pt",
          return_dict=True,
      )
      tokens = encoded["input_ids"].to(DEVICE)
    else:
        tokens = model.to_tokens(prompt)

    output_tokens = model.generate(
        tokens,
        return_type="tokens",
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=TEMP,
        stop_at_eos=True,
        eos_token_id=stop_ids,
        verbose=False,
    )

    new_tokens = output_tokens[0, tokens.shape[1]:]
    return tokenizer.decode(
        new_tokens.cpu(),
        skip_special_tokens=True,
    ).strip()


display(Markdown("## BASELINE (no steering)\n---"))

for prompt in prompts:
    output = generate_response(prompt)
    display(Markdown(f"---\n**> {prompt}**\n\n{output}"))

In [ ]:
display(Markdown(
    f"## STEERED (feature {ITALIAN_FEATURE}, strength={STRENGTH})\n---"
))

for prompt in prompts:
    with model.hooks(fwd_hooks=[(HOOK_POINT, hook_fn)]):
        output = generate_response(prompt)

    display(Markdown(f"---\n**> {prompt}**\n\n{output}"))

## 12. Sweep Gemma 3 steering strengths

Repeat the strength sweep for Gemma 3. The wider range deliberately explores both weak interventions and possible failure modes. Large values can produce cultural keyword repetition or degraded answers rather than meaningful control. Start with the smaller values and inspect coherence alongside any Italian references. Strengths cannot be directly calibrated against the Gemma 2 sweep.

In [ ]:
prompt = "Tell me about yourself."

for strength in [0, 50, 150, 250, 500, 1000, 2000, 5000, 7000, 10000]:
    sweep_hook = partial(
        steering_hook,
        feature_dir=feature_dir,
        strength=strength,
    )

    with model.hooks(fwd_hooks=[(HOOK_POINT, sweep_hook)]):
        output = generate_response(prompt, max_new_tokens=100)

    display(Markdown(f"### strength={strength}\n\n{output}"))

## 13. Record what your experiment showed

Write a short account of your hypothesis, prompt design, selected feature ID and SAE, and the strengths you tested for each model. Include examples that support your interpretation and examples where the effect was absent or the output became worse.

Distinguish Italian-language generation from mentions of Italian culture: a response that repeats “pasta” is different from a fluent Italian answer. The supplied cultural examples also include stereotypes, so findings should be described in terms of this dataset rather than as claims about Italian people.

For a follow-up, change one variable at a time: test another ranked feature, replace the discovery prompts, or expand the held-out evaluation set. If you tune strengths on the seven evaluation questions, use fresh questions for a final check. Record your package versions and generation settings so the run can be reproduced.